In [8]:
# ==========================================
# TRUE FUTURE FORECASTING MODEL (Lag-Based)
# ==========================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
import joblib

# ===============================
# STEP 1 — Load Data
# ===============================

df = pd.read_csv("clean_dwlr.csv")

print("Initial rows:", len(df))

# ===============================
# STEP 2 — Clean Data
# ===============================

df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df[df['date'].notna()]
df = df[df['currentlevel'].notna()]
df = df[np.isfinite(df['currentlevel'])]

# Remove extreme values
df = df[(df['currentlevel'] >= 0) & (df['currentlevel'] <= 200)]

print("Rows after cleaning:", len(df))

# ===============================
# STEP 3 — Sort Properly
# ===============================

df = df.sort_values(['station_name', 'date'])

# ===============================
# STEP 4 — Create Lag Features
# ===============================

df['lag1'] = df.groupby('station_name')['currentlevel'].shift(1)
df['lag2'] = df.groupby('station_name')['currentlevel'].shift(2)
df['lag3'] = df.groupby('station_name')['currentlevel'].shift(3)
df['lag4'] = df.groupby('station_name')['currentlevel'].shift(4)

# Target = next value (t+1)
df['target'] = df.groupby('station_name')['currentlevel'].shift(-1)

# Remove rows with NaN (created by shifting)
df = df.dropna()

print("Rows after lag creation:", len(df))

# ===============================
# STEP 5 — Time Features
# ===============================

df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

# ===============================
# STEP 6 — Encode Station
# ===============================

station_encoder = LabelEncoder()
df['station_encoded'] = station_encoder.fit_transform(df['station_name'])

# ===============================
# STEP 7 — Feature Selection
# ===============================

features = [
    'lag1',
    'lag2',
    'lag3',
    'lag4',
    'month',
    'year',
    'station_encoded'
]

X = df[features]
y = df['target']

print("Training samples:", len(X))

# ===============================
# STEP 8 — Train/Test Split (Time-Aware)
# ===============================

# IMPORTANT: do NOT shuffle for time-series
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# ===============================
# STEP 9 — Train Model
# ===============================

model = XGBRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# ===============================
# STEP 10 — Evaluate
# ===============================

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print("\n===============================")
print("FORECAST MODEL PERFORMANCE")
print("===============================")
print("MAE:", round(mae, 4))
print("R2 Score:", round(r2, 4))

# ===============================
# STEP 11 — Save Model
# ===============================

joblib.dump(model, "forecast_model.pkl")
joblib.dump(station_encoder, "station_encoder.pkl")

print("\nForecast model saved successfully!")

Initial rows: 550850
Rows after cleaning: 550841
Rows after lag creation: 443424
Training samples: 443424

FORECAST MODEL PERFORMANCE
MAE: 2.0782
R2 Score: 0.8156

Forecast model saved successfully!


In [11]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 3.2 MB/s eta 0:00:00


In [ ]:
from fastapi import FastAPI
import joblib
import groq
import numpy as np
from groq import Groq
import os

app = FastAPI()

# Load model & encoder
model = joblib.load("forecast_model.pkl")
station_encoder = joblib.load("station_encoder.pkl")

# Groq client
client = Groq(api_key="Groq_API_Key")

@app.post("/predict/{station_name}")
def predict(station_name: str):

    # Example lag values (you will fetch real ones from DB)
    lag_values = [5.0, 4.8, 4.6, 4.5]
    month = 7
    year = 2026

    station_encoded = station_encoder.transform([station_name])[0]

    features = np.array([[lag_values[0], lag_values[1], lag_values[2],
                          lag_values[3], month, year, station_encoded]])

    prediction = model.predict(features)[0]

    # Send prediction to Groq for explanation
    prompt = f"""
    Predicted groundwater level next month: {prediction}.
    Previous levels: {lag_values}.
    Analyze trend and risk.
    """

    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[{"role": "user", "content": prompt}]
    )

    explanation = response.choices[0].message.content

    return {
        "station": station_name,
        "predicted_next_level": float(prediction),
        "trend_analysis": explanation
    }